# Proyecto G14 - Produccion O&G '15-'15 USA

# Introducción

Utilizaremos un Dataset con la producción de petróleo y gas de pozos onshore y offshore de los Estados Unidos de América entre los años 2015-2025.

El objetivo es:

1 - realizar un modelo capaz de predecir la tendencia a futuro de la producción de petróleo y gas en Estados Unidos.

2 - realizar un modelo que compare la producción onshore vs offshore, que prediga la tendencia (más onshore u offshore).

Alguno de estos 2 objetivos.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    MinMaxScaler,
    StandardScaler,
    Normalizer,
    PowerTransformer,
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

print("Versión de pandas:", pd.__version__)

Versión de pandas: 3.0.5


In [6]:
df2=pd.read_csv("/Users/julianquinteros/Documents/Mi-Proyecto/Proyecto G14/OGORBcsv.csv")
df2

,Production Date,Land Class,Land Category,State,County,FIPS Code,Offshore Region,Commodity,Disposition Code,Disposition Description,Volume
0,01/01/2015,Federal,Offshore,NaN,NaN,NaN,Offshore Alaska,Gas (Mcf),1,Sales-Royalty Due-MEASURED,0
1,01/01/2015,Federal,Offshore,NaN,NaN,NaN,Offshore Gulf,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"14,508,756"
2,01/01/2015,Federal,Offshore,NaN,NaN,NaN,Offshore Pacific,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"1,207,520"
3,01/01/2015,Federal,Offshore,NaN,NaN,NaN,Offshore Gulf,Gas (Mcf),4,Sales-Royalty Due-Not Measured,"487,324"
4,01/01/2015,Federal,Offshore,NaN,NaN,NaN,Offshore Pacific,Gas (Mcf),4,Sales-Royalty Due-Not Measured,"114,808"
...,...,...,...,...,...,...,...,...,...,...,...
470825,05/01/2025,Native American,Onshore,NaN,NaN,NaN,NaN,Oil (bbl),62,Vented Gas Well Gas - Royalty Not Due,0
470826,05/01/2025,Native American,Onshore,NaN,NaN,NaN,NaN,Oil (bbl),66,Flared Oil Well Gas - Royalty Due,0
470827,05/01/2025,Native American,Onshore,NaN,NaN,NaN,NaN,Oil (bbl),67,Flared Gas Well Gas - Royalty Due,0
470828,05/01/2025,Native American,Onshore,NaN,NaN,NaN,NaN,Oil (bbl),68,Vented Oil Well Gas - Royalty Due,0


In [ ]:
# Contamos los datos nulos en cada columna del DataFrame df2
df2.isnull().sum()


Production Date                 0
Land Class                      0
Land Category                   0
State                       16630
County                      16630
FIPS Code                   16630
Offshore Region            460634
Commodity                       0
Disposition Code                0
Disposition Description         0
Volume                          0
dtype: int64

In [17]:
# Reemplazamos los valores nulos en "State" por "Sin datos"
df2.fillna({"State": "Sin datos"}, inplace=True)

,Production Date,Land Class,Land Category,State,County,FIPS Code,Offshore Region,Commodity,Disposition Code,Disposition Description,Volume
0,01/01/2015,Federal,Offshore,Sin datos,NaN,NaN,Offshore Alaska,Gas (Mcf),1,Sales-Royalty Due-MEASURED,0
1,01/01/2015,Federal,Offshore,Sin datos,NaN,NaN,Offshore Gulf,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"14,508,756"
2,01/01/2015,Federal,Offshore,Sin datos,NaN,NaN,Offshore Pacific,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"1,207,520"
3,01/01/2015,Federal,Offshore,Sin datos,NaN,NaN,Offshore Gulf,Gas (Mcf),4,Sales-Royalty Due-Not Measured,"487,324"
4,01/01/2015,Federal,Offshore,Sin datos,NaN,NaN,Offshore Pacific,Gas (Mcf),4,Sales-Royalty Due-Not Measured,"114,808"
...,...,...,...,...,...,...,...,...,...,...,...
470825,05/01/2025,Native American,Onshore,Sin datos,NaN,NaN,NaN,Oil (bbl),62,Vented Gas Well Gas - Royalty Not Due,0
470826,05/01/2025,Native American,Onshore,Sin datos,NaN,NaN,NaN,Oil (bbl),66,Flared Oil Well Gas - Royalty Due,0
470827,05/01/2025,Native American,Onshore,Sin datos,NaN,NaN,NaN,Oil (bbl),67,Flared Gas Well Gas - Royalty Due,0
470828,05/01/2025,Native American,Onshore,Sin datos,NaN,NaN,NaN,Oil (bbl),68,Vented Oil Well Gas - Royalty Due,0


In [ ]:
# Verificamos
df2.isnull().sum()

Production Date                 0
Land Class                      0
Land Category                   0
State                           0
County                      16630
FIPS Code                   16630
Offshore Region            460634
Commodity                       0
Disposition Code                0
Disposition Description         0
Volume                          0
dtype: int64

In [ ]:
# Revisamos que haya reemplazado correctamente los valores nulos en la columna "State"
df2.head(100)

,Production Date,Land Class,Land Category,State,County,FIPS Code,Offshore Region,Commodity,Disposition Code,Disposition Description,Volume
0,01/01/2015,Federal,Offshore,Sin datos,NaN,NaN,Offshore Alaska,Gas (Mcf),1,Sales-Royalty Due-MEASURED,0
1,01/01/2015,Federal,Offshore,Sin datos,NaN,NaN,Offshore Gulf,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"14,508,756"
2,01/01/2015,Federal,Offshore,Sin datos,NaN,NaN,Offshore Pacific,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"1,207,520"
3,01/01/2015,Federal,Offshore,Sin datos,NaN,NaN,Offshore Gulf,Gas (Mcf),4,Sales-Royalty Due-Not Measured,"487,324"
4,01/01/2015,Federal,Offshore,Sin datos,NaN,NaN,Offshore Pacific,Gas (Mcf),4,Sales-Royalty Due-Not Measured,"114,808"
...,...,...,...,...,...,...,...,...,...,...,...
95,01/01/2015,Federal,Onshore,AR,Logan,5083.0,NaN,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"302,347"
96,01/01/2015,Federal,Onshore,AR,Pope,5115.0,NaN,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"11,267"
97,01/01/2015,Federal,Onshore,AR,Sebastian,5131.0,NaN,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"211,689"
98,01/01/2015,Federal,Onshore,AR,Sharp,5135.0,NaN,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"2,178"


In [23]:
# Borramos County y FIPS Code porque no los vamos a usar, sería hilar demasiado fino.
df2.drop(columns=["County", "FIPS Code"], inplace=True)

In [25]:
# Reemplazamos los valores nulos en "Offshore Region" por "Onshore"
df2.fillna({"Offshore Region": "Onshore"}, inplace=True)

,Production Date,Land Class,Land Category,State,Offshore Region,Commodity,Disposition Code,Disposition Description,Volume
0,01/01/2015,Federal,Offshore,Sin datos,Offshore Alaska,Gas (Mcf),1,Sales-Royalty Due-MEASURED,0
1,01/01/2015,Federal,Offshore,Sin datos,Offshore Gulf,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"14,508,756"
2,01/01/2015,Federal,Offshore,Sin datos,Offshore Pacific,Gas (Mcf),1,Sales-Royalty Due-MEASURED,"1,207,520"
3,01/01/2015,Federal,Offshore,Sin datos,Offshore Gulf,Gas (Mcf),4,Sales-Royalty Due-Not Measured,"487,324"
4,01/01/2015,Federal,Offshore,Sin datos,Offshore Pacific,Gas (Mcf),4,Sales-Royalty Due-Not Measured,"114,808"
...,...,...,...,...,...,...,...,...,...
470825,05/01/2025,Native American,Onshore,Sin datos,Onshore,Oil (bbl),62,Vented Gas Well Gas - Royalty Not Due,0
470826,05/01/2025,Native American,Onshore,Sin datos,Onshore,Oil (bbl),66,Flared Oil Well Gas - Royalty Due,0
470827,05/01/2025,Native American,Onshore,Sin datos,Onshore,Oil (bbl),67,Flared Gas Well Gas - Royalty Due,0
470828,05/01/2025,Native American,Onshore,Sin datos,Onshore,Oil (bbl),68,Vented Oil Well Gas - Royalty Due,0


In [26]:
# Verificamos valores nulos en el DataFrame df2
df2.isnull().sum()

Production Date            0
Land Class                 0
Land Category              0
State                      0
Offshore Region            0
Commodity                  0
Disposition Code           0
Disposition Description    0
Volume                     0
dtype: int64